In [1]:
# ============================================================
# PS S6E5 - exp_20260501_006_cat_original_prior_min_without_normalized
# CatBoost + original prior minimal features
# No Normalized_TyreLife / No proxy / No original row merge
# ============================================================

In [2]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier, Pool

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260501_006_cat_original_prior_min_without_normalized"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    ORIG_BASE_CANDIDATES = [
        Path("/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction"),
    ]

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"
    DANGER_COL = "Normalized_TyreLife"

    SEED = 42
    N_SPLITS = 5

    PARAMS = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": 5000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "random_seed": SEED,
        "early_stopping_rounds": 300,
        "verbose": 300,
        "allow_writing_files": False,
    }


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Load competition data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

print("train shape:", train.shape)
print("test shape :", test.shape)
print("sub shape  :", sub.shape)

print("\ntrain columns:")
print(train.columns.tolist())

print("\ntest columns:")
print(test.columns.tolist())

assert CFG.ID_COL in train.columns
assert CFG.ID_COL in test.columns
assert CFG.TARGET in train.columns
assert CFG.TARGET not in test.columns

feature_cols_raw = [c for c in train.columns if c not in [CFG.ID_COL, CFG.TARGET]]

print("\nraw feature cols:", len(feature_cols_raw))
print(feature_cols_raw)

print("\ntarget distribution:")
print(train[CFG.TARGET].value_counts(dropna=False))
print(train[CFG.TARGET].value_counts(normalize=True, dropna=False))

train shape: (439140, 16)
test shape : (188165, 15)
sub shape  : (188165, 2)

train columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']

test columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

raw feature cols: 14
['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

target distribution:
PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


In [5]:
# ============================================================
# Locate and load original dataset
# ============================================================

all_csvs = sorted(Path("/kaggle/input").glob("**/*.csv"))

orig_base = None
for p in CFG.ORIG_BASE_CANDIDATES:
    if p.exists():
        orig_base = p
        break

if orig_base is not None:
    orig_csvs = sorted(orig_base.glob("*.csv"))
else:
    orig_csvs = [
        p for p in all_csvs
        if "playground-series-s6e5" not in str(p)
    ]

print("\noriginal csv candidates:")
for p in orig_csvs:
    print(p)

orig_infos = []
for p in orig_csvs:
    try:
        tmp = pd.read_csv(p, nrows=5)
        cols = tmp.columns.tolist()
        overlap = len((set(train.columns) - {CFG.ID_COL}) & set(cols))
        orig_infos.append({
            "path": str(p),
            "filename": p.name,
            "ncols": len(cols),
            "overlap": overlap,
            "has_target": CFG.TARGET in cols,
            "has_danger": CFG.DANGER_COL in cols,
            "columns": cols,
        })
    except Exception as e:
        print("failed:", p, e)

assert len(orig_infos) > 0, "original csv candidate was not found."

orig_infos = sorted(
    orig_infos,
    key=lambda x: (x["has_target"], x["overlap"], x["ncols"]),
    reverse=True,
)

orig_path = Path(orig_infos[0]["path"])
print("\nselected original:", orig_path)

orig = pd.read_csv(orig_path)

print("original shape:", orig.shape)
print("original columns:")
print(orig.columns.tolist())

assert CFG.TARGET in orig.columns, "originalにtargetがありません"
assert CFG.DANGER_COL in orig.columns, "想定ではNormalized_TyreLifeがoriginalに存在するはずです"

# 危険列は即削除。以降の処理で絶対に参照しない。
orig = orig.drop(columns=[CFG.DANGER_COL])

print("\noriginal columns after dropping danger col:")
print(orig.columns.tolist())

train_cols_no_id = set(train.columns) - {CFG.ID_COL}
orig_cols = set(orig.columns)

print("\ntrain only vs original after dropping danger:")
print(sorted(train_cols_no_id - orig_cols))

print("\noriginal only vs train after dropping danger:")
print(sorted(orig_cols - train_cols_no_id))

assert sorted(train_cols_no_id - orig_cols) == [], "train側にoriginalへ対応しない列があります"
assert sorted(orig_cols - train_cols_no_id) == [], "danger除外後もoriginal側に余分な列があります"


original csv candidates:
/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv

selected original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
original shape: (101371, 16)
original columns:
['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'Race', 'Year', 'LapTime_Delta', 'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress', 'Normalized_TyreLife', 'Position_Change']

original columns after dropping danger col:
['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'Race', 'Year', 'LapTime_Delta', 'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress', 'Position_Change']

train only vs original after dropping danger:
[]

original only vs train after dropping danger:
[]


In [6]:
# ============================================================
# Feature Engineering: original prior min
# ============================================================

def add_original_prior_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    orig_df: pd.DataFrame,
    target_col: str,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], dict]:
    """
    Add minimal original target prior features.

    Safe policy:
    - Use original target only.
    - Do not use competition train target.
    - Do not use Normalized_TyreLife.
    - Do not merge original rows into training.
    - Coarse priors only.
    """

    train_df = train_df.copy()
    test_df = test_df.copy()
    orig_df = orig_df.copy()

    global_prior = float(orig_df[target_col].mean())
    print("\noriginal global prior:", global_prior)
    print("competition train target mean:", float(train_df[target_col].mean()))

    added = []
    prior_info = {
        "global_prior": global_prior,
        "categorical_prior_cols": [],
        "numeric_bin_prior_cols": [],
        "pair_prior_cols": [],
    }

    # ----------------------------
    # 1. Categorical original priors
    # ----------------------------
    cat_prior_cols = [
        "Driver",
        "Compound",
        "Race",
        "Year",
        "Stint",
        "PitStop",
        "Position",
    ]

    for col in cat_prior_cols:
        if col not in orig_df.columns:
            continue
        if col not in train_df.columns or col not in test_df.columns:
            continue

        prior_name = f"ORIG_PRIOR_{col}"

        prior = (
            orig_df
            .groupby(col, dropna=False)[target_col]
            .agg(["mean", "count"])
            .reset_index()
        )
        prior = prior.rename(columns={
            "mean": prior_name,
            "count": f"{prior_name}_count",
        })

        train_df = train_df.merge(prior, on=col, how="left")
        test_df = test_df.merge(prior, on=col, how="left")

        train_df[prior_name] = train_df[prior_name].fillna(global_prior)
        test_df[prior_name] = test_df[prior_name].fillna(global_prior)

        train_df[f"{prior_name}_count"] = train_df[f"{prior_name}_count"].fillna(0)
        test_df[f"{prior_name}_count"] = test_df[f"{prior_name}_count"].fillna(0)

        added.extend([prior_name, f"{prior_name}_count"])
        prior_info["categorical_prior_cols"].append(col)

    # ----------------------------
    # 2. Numeric bin original priors
    # ----------------------------
    numeric_bin_cols = [
        "LapNumber",
        "TyreLife",
        "RaceProgress",
        "Cumulative_Degradation",
        "LapTime_Delta",
    ]

    combined_for_edges = pd.concat(
        [train_df[numeric_bin_cols], test_df[numeric_bin_cols], orig_df[numeric_bin_cols]],
        axis=0,
        ignore_index=True,
    )

    bin_edges_info = {}

    for col in numeric_bin_cols:
        if col not in orig_df.columns:
            continue
        if col not in train_df.columns or col not in test_df.columns:
            continue

        # qcutだとtrain/test/origでbin境界がズレるので、全feature分布から固定edgeを作る
        values = combined_for_edges[col].dropna()
        if values.nunique() <= 2:
            continue

        try:
            quantiles = np.linspace(0, 1, 11)
            edges = np.unique(values.quantile(quantiles).values)

            if len(edges) <= 2:
                continue

            edges[0] = -np.inf
            edges[-1] = np.inf

            bin_col = f"{col}_bin10"
            prior_name = f"ORIG_PRIOR_{bin_col}"

            train_df[bin_col] = pd.cut(train_df[col], bins=edges, include_lowest=True)
            test_df[bin_col] = pd.cut(test_df[col], bins=edges, include_lowest=True)
            orig_df[bin_col] = pd.cut(orig_df[col], bins=edges, include_lowest=True)

            prior = (
                orig_df
                .groupby(bin_col, observed=True)[target_col]
                .agg(["mean", "count"])
                .reset_index()
            )
            prior = prior.rename(columns={
                "mean": prior_name,
                "count": f"{prior_name}_count",
            })

            train_df = train_df.merge(prior, on=bin_col, how="left")
            test_df = test_df.merge(prior, on=bin_col, how="left")

            train_df[prior_name] = train_df[prior_name].fillna(global_prior)
            test_df[prior_name] = test_df[prior_name].fillna(global_prior)

            train_df[f"{prior_name}_count"] = train_df[f"{prior_name}_count"].fillna(0)
            test_df[f"{prior_name}_count"] = test_df[f"{prior_name}_count"].fillna(0)

            # CatBoostにInterval dtypeを渡さないため、bin列は削除する
            train_df = train_df.drop(columns=[bin_col])
            test_df = test_df.drop(columns=[bin_col])
            orig_df = orig_df.drop(columns=[bin_col])

            added.extend([prior_name, f"{prior_name}_count"])
            prior_info["numeric_bin_prior_cols"].append(col)
            bin_edges_info[col] = [float(x) if np.isfinite(x) else str(x) for x in edges]

        except Exception as e:
            print(f"failed numeric bin prior for {col}:", e)

    prior_info["bin_edges"] = bin_edges_info

    # ----------------------------
    # 3. Very coarse pair priors
    # ----------------------------
    # 細かすぎる Race x Driver x Stint などは初回では入れない。
    pair_cols = [
        ("Compound", "Stint"),
        ("Compound", "PitStop"),
        ("Race", "Stint"),
        ("Race", "PitStop"),
    ]

    for c1, c2 in pair_cols:
        if c1 not in orig_df.columns or c2 not in orig_df.columns:
            continue
        if c1 not in train_df.columns or c2 not in train_df.columns:
            continue

        pair_key = f"{c1}__{c2}"
        prior_name = f"ORIG_PRIOR_{pair_key}"

        for df in [train_df, test_df, orig_df]:
            df[pair_key] = df[c1].astype(str) + "__" + df[c2].astype(str)

        prior = (
            orig_df
            .groupby(pair_key, dropna=False)[target_col]
            .agg(["mean", "count"])
            .reset_index()
        )
        prior = prior.rename(columns={
            "mean": prior_name,
            "count": f"{prior_name}_count",
        })

        train_df = train_df.merge(prior, on=pair_key, how="left")
        test_df = test_df.merge(prior, on=pair_key, how="left")

        train_df[prior_name] = train_df[prior_name].fillna(global_prior)
        test_df[prior_name] = test_df[prior_name].fillna(global_prior)

        train_df[f"{prior_name}_count"] = train_df[f"{prior_name}_count"].fillna(0)
        test_df[f"{prior_name}_count"] = test_df[f"{prior_name}_count"].fillna(0)

        # pair_key自体はカテゴリ特徴として残さず削除する
        train_df = train_df.drop(columns=[pair_key])
        test_df = test_df.drop(columns=[pair_key])
        orig_df = orig_df.drop(columns=[pair_key])

        added.extend([prior_name, f"{prior_name}_count"])
        prior_info["pair_prior_cols"].append([c1, c2])

    # ----------------------------
    # 4. Simple prior deviations
    # ----------------------------
    # globalとの差。target率差があるので、絶対値より相対差の方が効く可能性を見る。
    prior_mean_cols = [
        c for c in added
        if c.startswith("ORIG_PRIOR_") and not c.endswith("_count")
    ]

    for c in prior_mean_cols:
        new_col = f"{c}_minus_global"
        train_df[new_col] = train_df[c] - global_prior
        test_df[new_col] = test_df[c] - global_prior
        added.append(new_col)

    prior_info["added_features"] = added
    prior_info["added_feature_count"] = len(added)

    return train_df, test_df, added, prior_info


train_fe, test_fe, added_features, prior_info = add_original_prior_features(
    train_df=train,
    test_df=test,
    orig_df=orig,
    target_col=CFG.TARGET,
)

print("\nadded features:", len(added_features))
for c in added_features:
    print(c)

print("\nprior info:")
print(json.dumps(prior_info, ensure_ascii=False, indent=2)[:5000])

feature_cols = [c for c in train_fe.columns if c not in [CFG.ID_COL, CFG.TARGET]]
print("\ntotal feature cols:", len(feature_cols))


original global prior: 0.25479673673930414
competition train target mean: 0.19898210137996994

added features: 48
ORIG_PRIOR_Driver
ORIG_PRIOR_Driver_count
ORIG_PRIOR_Compound
ORIG_PRIOR_Compound_count
ORIG_PRIOR_Race
ORIG_PRIOR_Race_count
ORIG_PRIOR_Year
ORIG_PRIOR_Year_count
ORIG_PRIOR_Stint
ORIG_PRIOR_Stint_count
ORIG_PRIOR_PitStop
ORIG_PRIOR_PitStop_count
ORIG_PRIOR_Position
ORIG_PRIOR_Position_count
ORIG_PRIOR_LapNumber_bin10
ORIG_PRIOR_LapNumber_bin10_count
ORIG_PRIOR_TyreLife_bin10
ORIG_PRIOR_TyreLife_bin10_count
ORIG_PRIOR_RaceProgress_bin10
ORIG_PRIOR_RaceProgress_bin10_count
ORIG_PRIOR_Cumulative_Degradation_bin10
ORIG_PRIOR_Cumulative_Degradation_bin10_count
ORIG_PRIOR_LapTime_Delta_bin10
ORIG_PRIOR_LapTime_Delta_bin10_count
ORIG_PRIOR_Compound__Stint
ORIG_PRIOR_Compound__Stint_count
ORIG_PRIOR_Compound__PitStop
ORIG_PRIOR_Compound__PitStop_count
ORIG_PRIOR_Race__Stint
ORIG_PRIOR_Race__Stint_count
ORIG_PRIOR_Race__PitStop
ORIG_PRIOR_Race__PitStop_count
ORIG_PRIOR_Driver_min

In [7]:
# ============================================================
# Column types
# ============================================================

# 002 CatBoost RAW baseline に合わせる。
cat_cols = [
    "Driver",
    "Compound",
    "Race",
    "Year",
    "LapNumber",
    "PitStop",
    "Position",
    "Stint",
]
cat_cols = [c for c in cat_cols if c in feature_cols]

num_cols = [c for c in feature_cols if c not in cat_cols]

print("\ncat_cols:", len(cat_cols), cat_cols)
print("num_cols:", len(num_cols), num_cols)


cat_cols: 8 ['Driver', 'Compound', 'Race', 'Year', 'LapNumber', 'PitStop', 'Position', 'Stint']
num_cols: 54 ['TyreLife', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'ORIG_PRIOR_Driver', 'ORIG_PRIOR_Driver_count', 'ORIG_PRIOR_Compound', 'ORIG_PRIOR_Compound_count', 'ORIG_PRIOR_Race', 'ORIG_PRIOR_Race_count', 'ORIG_PRIOR_Year', 'ORIG_PRIOR_Year_count', 'ORIG_PRIOR_Stint', 'ORIG_PRIOR_Stint_count', 'ORIG_PRIOR_PitStop', 'ORIG_PRIOR_PitStop_count', 'ORIG_PRIOR_Position', 'ORIG_PRIOR_Position_count', 'ORIG_PRIOR_LapNumber_bin10', 'ORIG_PRIOR_LapNumber_bin10_count', 'ORIG_PRIOR_TyreLife_bin10', 'ORIG_PRIOR_TyreLife_bin10_count', 'ORIG_PRIOR_RaceProgress_bin10', 'ORIG_PRIOR_RaceProgress_bin10_count', 'ORIG_PRIOR_Cumulative_Degradation_bin10', 'ORIG_PRIOR_Cumulative_Degradation_bin10_count', 'ORIG_PRIOR_LapTime_Delta_bin10', 'ORIG_PRIOR_LapTime_Delta_bin10_count', 'ORIG_PRIOR_Compound__Stint', 'ORIG_PRIOR_Compound__Stint_count', 'ORIG_PRIOR_Co

In [8]:
# ============================================================
# Prepare X/y
# ============================================================

X = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()

# CatBoost categorical columns must be string-like with missing filled.
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("__MISSING__")
    X_test[c] = X_test[c].astype("string").fillna("__MISSING__")

assert train_fe[CFG.TARGET].notna().all()
assert set(train_fe[CFG.TARGET].dropna().unique()).issubset({0, 1, 0.0, 1.0})
y = train_fe[CFG.TARGET].astype(int).values

cat_features = [X.columns.get_loc(c) for c in cat_cols]

print("\nX shape:", X.shape)
print("X_test shape:", X_test.shape)
print("cat_features indices:", cat_features)

print("\nNaN count in X numeric part:", int(X[num_cols].isna().sum().sum()))
print("NaN count in X_test numeric part:", int(X_test[num_cols].isna().sum().sum()))


X shape: (439140, 62)
X_test shape: (188165, 62)
cat_features indices: [0, 1, 2, 3, 5, 4, 8, 6]

NaN count in X numeric part: 0
NaN count in X_test numeric part: 0


In [9]:
# ============================================================
# CV Training
# ============================================================

oof = np.zeros(len(train_fe), dtype=float)
pred = np.zeros(len(test_fe), dtype=float)

fold_scores = []
fold_best_iterations = []
feature_importance_list = []

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("\n" + "=" * 80)
    print(f"Fold {fold}")
    print("=" * 80)

    X_tr = X.iloc[tr_idx]
    X_va = X.iloc[va_idx]
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    valid_pool = Pool(X_va, y_va, cat_features=cat_features)
    test_pool = Pool(X_test, cat_features=cat_features)

    model = CatBoostClassifier(**CFG.PARAMS)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    va_pred = model.predict_proba(valid_pool)[:, 1]
    te_pred = model.predict_proba(test_pool)[:, 1]

    oof[va_idx] = va_pred
    pred += te_pred / CFG.N_SPLITS

    auc = roc_auc_score(y_va, va_pred)
    fold_scores.append(float(auc))
    fold_best_iterations.append(int(model.get_best_iteration()))

    print(f"fold {fold} AUC: {auc:.8f}")
    print(f"fold {fold} best_iteration: {model.get_best_iteration()}")

    fi = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.get_feature_importance(),
        "fold": fold,
    })
    feature_importance_list.append(fi)


cv_auc = roc_auc_score(y, oof)

print("\n" + "=" * 80)
print("CV RESULT")
print("=" * 80)
print("CV AUC:", cv_auc)
print("fold_scores:", fold_scores)
print("mean:", np.mean(fold_scores))
print("std :", np.std(fold_scores))
print("fold_best_iterations:", fold_best_iterations)


Fold 1
0:	test: 0.9071323	best: 0.9071323 (0)	total: 360ms	remaining: 29m 58s
300:	test: 0.9379517	best: 0.9379517 (300)	total: 1m 14s	remaining: 19m 27s
600:	test: 0.9425148	best: 0.9425148 (600)	total: 2m 28s	remaining: 18m 4s
900:	test: 0.9449996	best: 0.9449996 (900)	total: 3m 42s	remaining: 16m 50s
1200:	test: 0.9463084	best: 0.9463084 (1200)	total: 4m 56s	remaining: 15m 39s
1500:	test: 0.9472017	best: 0.9472017 (1500)	total: 6m 11s	remaining: 14m 25s
1800:	test: 0.9478168	best: 0.9478168 (1800)	total: 7m 26s	remaining: 13m 12s
2100:	test: 0.9482803	best: 0.9482803 (2100)	total: 8m 40s	remaining: 11m 58s
2400:	test: 0.9486037	best: 0.9486037 (2400)	total: 9m 55s	remaining: 10m 44s
2700:	test: 0.9488565	best: 0.9488565 (2700)	total: 11m 11s	remaining: 9m 31s
3000:	test: 0.9491301	best: 0.9491304 (2994)	total: 12m 28s	remaining: 8m 18s
3300:	test: 0.9493238	best: 0.9493238 (3300)	total: 13m 45s	remaining: 7m 4s
3600:	test: 0.9494977	best: 0.9494978 (3598)	total: 15m	remaining: 5m 4

In [10]:
# ============================================================
# Save artifacts
# ============================================================

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred)

sub_out = sub.copy()
pred_col = [c for c in sub_out.columns if c != CFG.ID_COL][0]
sub_out[pred_col] = pred
sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub_out.to_csv(sub_path, index=False)

feature_df = pd.DataFrame({
    "feature": feature_cols,
    "is_added": [c in added_features for c in feature_cols],
    "is_categorical": [c in cat_cols for c in feature_cols],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

fi_all = pd.concat(feature_importance_list, axis=0, ignore_index=True)
fi_summary = (
    fi_all
    .groupby("feature", as_index=False)["importance"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

fi_all.to_csv(CFG.OUTDIR / "feature_importance_by_fold.csv", index=False)
fi_summary.to_csv(CFG.OUTDIR / "feature_importance.csv", index=False)

with open(CFG.OUTDIR / "prior_info.json", "w", encoding="utf-8") as f:
    json.dump(prior_info, f, ensure_ascii=False, indent=2)

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "data": {
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "original_shape_after_danger_drop": list(orig.shape),
        "selected_original_path": str(orig_path),
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "competition_target_mean": float(train[CFG.TARGET].mean()),
        "original_target_mean_after_danger_drop": float(orig[CFG.TARGET].mean()),
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
    },
    "features": {
        "raw_feature_count": len(feature_cols_raw),
        "total_feature_count": len(feature_cols),
        "added_feature_count": len(added_features),
        "categorical_features": cat_cols,
        "numerical_features": num_cols,
        "added_features": added_features,
        "prior_info": prior_info,
        "notes": [
            "Original dataset is used only for target prior features.",
            "Normalized_TyreLife is dropped before any feature engineering.",
            "No Normalized_TyreLife proxy is created.",
            "Original rows are not merged into training.",
            "Competition train target is not used for target encoding.",
            "Coarse categorical, numeric-bin, and pair priors only.",
        ],
    },
    "model": {
        "family": "CatBoost",
        "params": CFG.PARAMS,
        "cv": {
            "scheme": "StratifiedKFold",
            "n_splits": CFG.N_SPLITS,
            "shuffle": True,
            "random_state": CFG.SEED,
        },
    },
    "result": {
        "cv_auc": float(cv_auc),
        "fold_scores": fold_scores,
        "fold_best_iterations": fold_best_iterations,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
    },
    "artifacts": {
        "oof": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "submission": str(sub_path),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "feature_importance": str(CFG.OUTDIR / "feature_importance.csv"),
        "prior_info": str(CFG.OUTDIR / "prior_info.json"),
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved artifacts to:", CFG.OUTDIR)
print("Submission:", sub_path)

display(fi_summary.head(60))
display(sub_out.head())


Saved artifacts to: /kaggle/working/exp_20260501_006_cat_original_prior_min_without_normalized
Submission: /kaggle/working/exp_20260501_006_cat_original_prior_min_without_normalized/submission_exp_20260501_006_cat_original_prior_min_without_normalized.csv


,index,feature,mean,std
61,61,Year,22.113476,4.238495
51,51,ORIG_PRIOR_Year,8.433405,3.261223
53,53,ORIG_PRIOR_Year_minus_global,6.866500,2.206450
60,60,TyreLife,6.619832,0.141736
58,58,RaceProgress,4.605447,0.240608
1,1,Cumulative_Degradation,2.748353,0.133833
56,56,Position_Change,2.727073,0.158005
5,5,LapTime_Delta,2.718948,0.187496
10,10,ORIG_PRIOR_Compound__Stint,2.684419,0.557167
12,12,ORIG_PRIOR_Compound__Stint_minus_global,2.619536,0.599810


,id,PitNextLap
0,439140,0.006255
1,439141,0.004372
2,439142,0.005528
3,439143,0.146005
4,439144,0.814161
